# Model ownership

In PyTorch, freezing a model and then training with a handle you already hold is a silent no-op:

```python
for p in model.parameters(): p.requires_grad = False
optimizer.step()   # references the same params: no error, and no training
```

The loop runs, the loss is computed, and the model never learns. The problem is a *stale reference to something that changed*: `requires_grad = False` mutates state in the C++ runtime while every Python reference looks unchanged.

In idris-ml a model is a **linear resource**: `forward`, `eval`, and `freeze` each *consume* the handle and return a fresh one, and the compiler counts uses. This notebook shows the counting in action.

## The multiplicity is in the signatures

The `(1 _ : ...)` argument means the model must be used exactly once:

In [1]:
:t eval

Ml.Nn.Module.eval : UserExecutorTraining ex => Params l => (1 _ : l i o ex dt WithGrad) -> L IO (l i o ex dt NoGrad)


In [2]:
:t freeze

Ml.Nn.Module.freeze : UserExecutorTraining ex => Params l => (1 _ : l i o ex dt g) -> L IO (Frozen (l i o ex dt g))


## Consuming a model and using the fresh handle

`eval` takes the model out of training (`WithGrad` becomes `NoGrad`) and returns the new handle. Using the returned handle is the ordinary path:

In [3]:
:exec run (do {
  model <- runInitL (linear {i=2} {o=3} {ex=TapeExecutor} {dt=F64} {g=WithGrad});
  infer <- eval model;
  x <- liftIO1 (tensor {dims=[3,2]} {ex=TapeExecutor} {dt=F64} (FromVect [0.0,0.0, 1.0,1.0, 2.0,0.0]));
  (MkBang out # infer1) <- forward {b=3} infer x;
  discard infer1;
  liftIO1 (putStrLn "consumed `model` once; the fresh handle `infer` forwards fine") })

consumed `model` once; the fresh handle `infer` forwards fine


## Reusing a consumed handle is a compile error

The freeze-then-train bug: consume `model`, then use the old name again. The next cell is *expected to fail* with a linearity error ("There are 2 uses of linear name model"):

In [4]:
:exec run (do {
  model <- runInitL (linear {i=2} {o=3} {ex=TapeExecutor} {dt=F64} {g=WithGrad});
  m1 <- eval model;
  m2 <- eval model;
  discard m1;
  discard m2;
  liftIO1 (putStrLn "should not reach here") })

Error: There are 2 uses of linear name model. 

(Interactive):1:17--1:22
 1 | :exec run (do { model <- runInitL (linear {i=2} {o=3} {ex=TapeExecutor} {dt=F64} {g=WithGrad}); m1 <- eval model; m2 <- eval model; discard m1; discard m2; liftIO1 (putStrLn "should not reach here") })
                     ^^^^^

Suggestion: linearly bounded variables must be used exactly once.


## An inference model's output can't reach the optimizer

The grad-mode half of the guarantee: `eval` returns a `NoGrad` model, its outputs produce a `NoGrad` loss, and `trainStep` only accepts a `WithGrad` loss. The next cell is *expected to fail*:

In [5]:
:exec run (do {
  model <- runInitL (linear {i=2} {o=3} {ex=TapeExecutor} {dt=F64} {g=WithGrad});
  opt <- liftIO1 (sgd 0.1 defaultOpts);
  infer <- eval model;
  x <- liftIO1 (tensor {dims=[3,2]} {ex=TapeExecutor} {dt=F64} (FromVect [0.0,0.0, 1.0,1.0, 2.0,0.0]));
  y <- liftIO1 (tensor {dims=[3,3]} {ex=TapeExecutor} {dt=F64} (FromVect [1.0,0.0,0.0, 0.0,1.0,0.0, 0.0,0.0,1.0]));
  (MkBang out # infer1) <- forward {b=3} infer x;
  loss <- tnllLossMeanL {b=3} {n=3} out (retypeGrad y);
  _ <- liftIO1 (trainStep opt loss);
  discard infer1;
  liftIO1 (putStrLn "should not reach here") })

Error: When unifying:
    Tensor [] TapeExecutor (Float 64) NoGrad
and:
    Tensor [] ?ex ?dt WithGrad
Mismatch between: NoGrad and WithGrad.

(Interactive):1:502--1:506
 1 | :exec run (do { model <- runInitL (linear {i=2} {o=3} {ex=TapeExecutor} {dt=F64} {g=WithGrad}); opt <- liftIO1 (sgd 0.1 defaultOpts); infer <- eval model; x <- liftIO1 (tensor {dims=[3,2]} {ex=TapeExecutor} {dt=F64} (FromVect [0.0,0.0, 1.0,1.0, 2.0,0.0])); y <- liftIO1 (tensor {dims=[3,3]} {ex=TapeExecutor} {dt=F64} (FromVect [1.0,0.0,0.0, 0.0,1.0,0.0, 0.0,0.0,1.0])); (MkBang out # infer1) <- forward {b=3} infer x; loss <- tnllLossMeanL {b=3} {n=3} out (retypeGrad y); _ <- liftIO1 (trainStep opt loss); discard infer1; liftIO1 (putStrLn "should not reach here") })
                                                                                                                                                                                                                                                               

## Tensors stay unrestricted

Individual tensors are deliberately *not* linear: reverse-mode autograd needs the same tensor feeding several branches of the graph, so use-exactly-once typing on tensors would reject correct programs. The linear rule applies only to the model handle, the value the stale-reference bug is about.

Next: [06 Sequences](06_sequences.ipynb) — recurrent models for time-series data.